# Suicide Risk Prediction with SHAP Explanations

**Based on:** *AI in Modern Psychology*, Chapter 8 (Diagnostic Applications) & Chapter 11 (Clinical Decision Making).

⚠️ **Clinical disclaimer:** This is a simulated educational example with synthetic data. Do **not** use for real risk assessment.


## Learning Objectives

- Build a gradient-boosting model on synthetic EHR data.
- Apply SHAP values for global and local explanation.
- Discuss ethical trade-offs: false positives, calibration, bias, and clinical use.


## Setup


In [ ]:
# !pip install xgboost shap


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import shap
from code.clinical.risk_prediction import generate_synthetic_ehr, FEATURE_COLS
np.random.seed(42)


## 1. Generate synthetic EHR data


In [ ]:
df = generate_synthetic_ehr(10_000, seed=42)
print(f'Base rate of suicide attempt: {df["attempt"].mean():.3%}')
df.head()


## 2. Train / test split


In [ ]:
X = df[FEATURE_COLS]; y = df['attempt']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = X_train.copy(); X_test_s = X_test.copy()
X_train_s[['age', 'ed_visits']] = scaler.fit_transform(X_train[['age', 'ed_visits']])
X_test_s[['age', 'ed_visits']]  = scaler.transform(X_test[['age', 'ed_visits']])
print(f'Train: {len(X_train)} (attempts: {y_train.sum()})')
print(f'Test:  {len(X_test)} (attempts: {y_test.sum()})')


## 3. Train XGBoost


In [ ]:
model = xgb.XGBClassifier(
    n_estimators=100, max_depth=4, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    scale_pos_weight=(len(y_train) - y_train.sum()) / max(y_train.sum(), 1),
    eval_metric='logloss'
)
model.fit(X_train_s, y_train)
y_proba = model.predict_proba(X_test_s)[:, 1]
y_pred  = (y_proba > 0.5).astype(int)
print(f'AUC: {roc_auc_score(y_test, y_proba):.3f}')
print(classification_report(y_test, y_pred, target_names=['No attempt', 'Attempt']))


## 4. SHAP — global feature importance


In [ ]:
explainer   = shap.Explainer(model, X_train_s)
shap_values = explainer(X_test_s)
shap.summary_plot(shap_values, X_test_s, feature_names=FEATURE_COLS, show=False)
plt.tight_layout(); plt.show()


## 5. Individual explanation


In [ ]:
hi_idx = np.where(y_proba > 0.6)[0]
if len(hi_idx):
    i = int(hi_idx[0])
    print(f'Patient index: {i}  predicted risk: {y_proba[i]:.2%}')
    shap.waterfall_plot(shap_values[i], max_display=8, show=False)
    plt.tight_layout(); plt.show()
else:
    print('No high-risk patients in this slice; lower the threshold.')


## 6. Ethical Considerations

With a 0.5 % base rate, even a strong model produces many false positives. The book argues for **AI-calibrated clinical judgment** rather than automation. See `ethics_toolkit/algorithmic_recourse_protocol.md` for how to operationalize patient recourse.

**Reflection prompts:**

1. Would you deploy this model in your clinic? Why or why not?
2. How would you communicate a high-risk prediction without causing undue distress?
3. What demographic audits are missing from this notebook?


## References

- Walsh, C. G., Ribeiro, J. D., & Franklin, J. C. (2017). Predicting risk of suicide attempts over time. *Clinical Psychological Science*, 5(3), 457–469.
- Lundberg, S. M., & Lee, S.-I. (2017). A unified approach to interpreting model predictions. *NeurIPS*.
